# Workshop self-test: *Swarm in the Lab*

**BiocAsia 2026 · satellite of ABACBS 2026 · RMIT University**

This notebook checks that your workshop machine (your RAIL instance) is ready for the afternoon. We also run it at the start of the workshop.

**How to run it**
1. In VS Code, connect to your RAIL instance (the **><** button in the bottom-left corner → *Connect to Host*).
2. Open this notebook and pick the **workshop Python kernel** in the top-right corner if VS Code asks.
3. Click **Run All**.

It takes about two minutes and costs less than one US cent in API calls. It doesn't change anything on your machine, apart from saving its result to `~/preflight_result.json`.

**Reading the results:** ✅ ready · ⚠️ works, but worth a look · ❌ needs fixing · ⏭️ skipped

The **last cell** prints one line starting with `PREFLIGHT`. If it says **READY**, you're all set. If it doesn't, reply to the setup email with that line (or a screenshot), or show it to a helper on the day.

## Settings
*Participants: you don't need to change anything here.*
*Organisers: fill in the `[PLACEHOLDER]` values once, before you copy this notebook to the instances.*

In [ ]:
from pathlib import Path

# --- Workshop materials on the RAIL instance --------------------------------
WORKSHOP_DIR       = Path.home() / "workshop"                     # [PLACEHOLDER]
DATASET_PATH       = WORKSHOP_DIR / "data" / "workshop.h5ad"      # [PLACEHOLDER] dataset used in Module 2
PAPER2AGENT_DIR    = WORKSHOP_DIR / "Paper2Agent"                 # [PLACEHOLDER]
VIRTUAL_BIOTECH_DIR = WORKSHOP_DIR / "virtual-biotech"            # [PLACEHOLDER]
REQUIRED_NOTEBOOKS = [                                            # [PLACEHOLDER] leave [] to skip
    # "01_virtual_lab.ipynb", "02_paper2agent.ipynb", "03_virtual_biotech.ipynb", "04_checking.ipynb",
]

# --- The running question ----------------------------------------------------
TARGET_GENE_SYMBOL = "CD276"            # B7-H3
TARGET_ENSEMBL_ID  = "ENSG00000103855"
TRIAL_TERM         = "B7-H3 OR CD276"
TRIAL_CONDITION    = "lung cancer"

# --- Models used for the "hello agent" checks --------------------------------
OPENAI_MODEL = "gpt-5.2"   # the model Virtual Lab will use in Module 1 (its default at the time of writing)
CLAUDE_MODEL = "haiku"     # cheap Agent SDK check; Virtual Biotech may use a larger model on the day

# --- Pre-built Paper2Agent MCP servers (Module 2) ----------------------------
# name -> hosted URL (https://.../mcp) or path to a local server .py file. Leave {} to skip.
PAPER_AGENT_SERVERS = {
    # "scanpy": "https://[PLACEHOLDER]/scanpy/mcp",
    # "tissue": "https://[PLACEHOLDER]/tissue/mcp",
}

TIMEOUT_S = 30   # per network request

In [ ]:
# Helper functions. No need to read this cell.
import asyncio, json, os, platform, shutil, socket, sys, tempfile, time
import urllib.parse, urllib.request
from importlib import metadata

RESULTS = []
ICONS = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌", "SKIP": "⏭️"}

def record(name, status, detail="", fix=""):
    # Re-running a cell replaces its earlier result instead of duplicating it.
    RESULTS[:] = [r for r in RESULTS if r["check"] != name]
    fix = fix if status in ("WARN", "FAIL") else ""
    RESULTS.append({"check": name, "status": status, "detail": detail, "fix": fix})
    print(f"{ICONS[status]} {name}: {detail}")
    if fix:
        print(f"      ↳ {fix}")

def _describe(e):
    return f"{type(e).__name__}: {str(e).strip().splitlines()[0][:200] if str(e).strip() else ''}"

MISSING_PKG = ("A package is missing from this kernel. Make sure the workshop kernel is selected (top-right); "
               "if it is, tell a helper.")

def _hint(fix, e):
    if isinstance(e, ImportError):
        return MISSING_PKG
    return fix(e) if callable(fix) else fix

def check(name, fn, fix="", on_error="FAIL"):
    """Run a check. fn() returns (status, detail). fix is a hint string, or a function of the exception."""
    try:
        status, detail = fn()
        hint = fix if isinstance(fix, str) else ""
    except Exception as e:
        status, detail = ("FAIL" if isinstance(e, ImportError) else on_error), _describe(e)
        hint = _hint(fix, e)
    record(name, status, detail, hint)

async def acheck(name, coro_fn, fix="", timeout=90, on_error="FAIL"):
    """Async version of check(), with a timeout."""
    try:
        status, detail = await asyncio.wait_for(coro_fn(), timeout)
        hint = fix if isinstance(fix, str) else ""
    except asyncio.TimeoutError:
        status, detail = on_error, f"no answer after {timeout}s"
        hint = _hint(fix, None)
    except Exception as e:
        status, detail = ("FAIL" if isinstance(e, ImportError) else on_error), _describe(e)
        hint = _hint(fix, e)
    record(name, status, detail, hint)

def http_json(url, params=None, payload=None, timeout=TIMEOUT_S):
    if params:
        url += "?" + urllib.parse.urlencode(params)
    data = json.dumps(payload).encode() if payload is not None else None
    headers = {"User-Agent": "workshop-preflight/1.0", "Accept": "application/json"}
    if data:
        headers["Content-Type"] = "application/json"
    req = urllib.request.Request(url, data=data, headers=headers)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())

def api_hint(e):
    """Turn an API error into a plain-language next step."""
    n = type(e).__name__ if e is not None else "Timeout"
    if "Authentication" in n or "PermissionDenied" in n:
        return "The API key was rejected. Tell a helper: the key on this machine may be wrong or disabled."
    if n == "NotFoundError":
        return "The model name wasn't recognised. Organisers: check the model setting in the Settings cell."
    if "RateLimit" in n:
        return "Rate-limited, or this machine's spending cap is used up. Wait a minute and re-run; if it persists, tell a helper."
    if "Connection" in n or "Timeout" in n:
        return "Couldn't reach the API. Check the machine's internet access, then re-run this cell."
    return "Re-run this cell once. If it fails again, tell a helper and show them this message."

HELPER = "Tell a helper. This is a machine setup problem, not something you did."
print("Helpers loaded.")

## 1 · The machine

In [ ]:
def _python():
    ok = sys.version_info >= (3, 12)   # Virtual Lab 1.2.0 uses f-string syntax that Python 3.11 can't parse
    return ("PASS" if ok else "FAIL"), f"Python {platform.python_version()} ({sys.executable})"
check("Python 3.12 or newer", _python,
      fix="Pick the workshop kernel in the top-right corner of the notebook, then Run All again. "
          "If it is already selected, tell a helper: Virtual Lab needs Python 3.12+.")

def _resources():
    cpus = os.cpu_count() or 0
    mem_gb = 0.0
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal:"):
                    mem_gb = int(line.split()[1]) / 1024**2
    except OSError:
        pass
    free_gb = shutil.disk_usage(Path.home()).free / 1024**3
    ok = cpus >= 4 and mem_gb >= 14 and free_gb >= 5
    return ("PASS" if ok else "WARN"), (f"{socket.gethostname()} · {cpus} vCPU · {mem_gb:.1f} GB RAM · "
                                        f"{free_gb:.1f} GB free in your home folder")
check("CPU, memory and disk", _resources,
      fix="Smaller than expected (4 vCPU, 16 GB RAM, 5+ GB free). It may still work. Mention it to a helper.")

def _desktop_ssh():
    # The Claude desktop app's SSH sessions (Module 3) need SFTP on the server and a home folder that allows executables.
    cfgs = [Path("/etc/ssh/sshd_config")]
    d = Path("/etc/ssh/sshd_config.d")
    if d.is_dir():
        cfgs += sorted(d.glob("*.conf"))
    readable, sftp = False, False
    for c in cfgs:
        try:
            for line in c.read_text().splitlines():
                s = line.strip().lower()
                if s.startswith("subsystem") and "sftp" in s:
                    sftp = True
            readable = True
        except OSError:
            pass
    home = str(Path.home().resolve())
    best_mnt, best_opts = "", ""
    try:
        with open("/proc/mounts") as f:
            for line in f:
                _, mnt, _, opts, *_ = line.split()
                inside = mnt == "/" or home == mnt or home.startswith(mnt.rstrip("/") + "/")
                if inside and len(mnt) > len(best_mnt):
                    best_mnt, best_opts = mnt, opts
    except OSError:
        pass
    noexec = "noexec" in best_opts.split(",")
    if noexec:
        return "FAIL", f"home folder is on a noexec mount ({best_mnt})"
    if not readable:
        return "WARN", "couldn't read the SSH server settings to confirm SFTP is on"
    if not sftp:
        return "FAIL", "the SSH server has no SFTP subsystem"
    return "PASS", "SFTP is on and your home folder allows the app's remote helper"
check("Ready for desktop-app SSH sessions (Module 3)", _desktop_ssh,
      fix="The desktop app may not connect in Module 3. You can use VS Code for that module instead. Tell a helper.")

## 2 · Software on the machine

In [ ]:
PACKAGES = [  # (label, import name, distribution name)
    ("virtual-lab (Module 1)",          "virtual_lab",      "virtual-lab"),
    ("openai",                          "openai",           "openai"),
    ("claude-agent-sdk (Module 3)",     "claude_agent_sdk", "claude-agent-sdk"),
    ("fastmcp (Modules 2-3)",           "fastmcp",          "fastmcp"),
    ("scanpy",                          "scanpy",           "scanpy"),
    ("anndata",                         "anndata",          "anndata"),
    ("cellxgene-census",                "cellxgene_census", "cellxgene-census"),
    ("pandas",                          "pandas",           "pandas"),
    ("biopython",                       "Bio",              "biopython"),
]

def _pkg(mod, dist):
    def fn():
        __import__(mod)
        try:
            v = metadata.version(dist)
        except metadata.PackageNotFoundError:
            v = "version unknown"
        return "PASS", v
    return fn

for label, mod, dist in PACKAGES:
    check(label, _pkg(mod, dist))

def _folders():
    missing = [str(p) for p in (WORKSHOP_DIR, PAPER2AGENT_DIR, VIRTUAL_BIOTECH_DIR) if not Path(p).expanduser().exists()]
    missing += [n for n in REQUIRED_NOTEBOOKS if not (WORKSHOP_DIR / n).exists()]
    if missing:
        return "FAIL", "missing: " + ", ".join(missing)
    return "PASS", f"workshop folder, Paper2Agent, Virtual Biotech and {len(REQUIRED_NOTEBOOKS)} module notebook(s) found"
check("Workshop materials", _folders, fix=HELPER)

## 3 · Hello, agent
Each framework talks to a model provider: **Virtual Lab** uses OpenAI, and **Virtual Biotech** uses Anthropic through the Claude Agent SDK. Both keys come pre-configured on your machine, with a spending cap. Each check below sends one tiny message.

In [ ]:
def _key(var, base_var):
    def fn():
        v = os.environ.get(var, "")
        if not v:
            return "FAIL", f"{var} is not set"
        base = os.environ.get(base_var)
        via = f", via gateway {base}" if base else ""
        return "PASS", f"set ({v[:6]}…, {len(v)} characters{via})"
    return fn
check("OpenAI key present", _key("OPENAI_API_KEY", "OPENAI_BASE_URL"), fix=HELPER)
check("Anthropic key present", _key("ANTHROPIC_API_KEY", "ANTHROPIC_BASE_URL"), fix=HELPER)

def _vlab_objects():
    from virtual_lab.agent import Agent
    from virtual_lab.run_meeting import run_meeting  # noqa: F401  (import check only)
    pi = Agent(title="Principal Investigator", expertise="tumour immunology",
               goal="check the workshop setup", role="lead the meeting", model=OPENAI_MODEL)
    return "PASS", f"created a test agent ({getattr(pi, 'title', 'PI')}, {OPENAI_MODEL}); run_meeting is available"
check("Virtual Lab agents can be created", _vlab_objects,
      fix="The virtual-lab package may be a different version from the one the notebooks expect. Tell a helper.")

def _openai_hello():
    from openai import OpenAI
    client = OpenAI(timeout=TIMEOUT_S * 2, max_retries=1)
    t0 = time.time()
    r = client.chat.completions.create(   # the same API Virtual Lab calls
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": "Reply with exactly two words: hello agent"}],
    )
    text = (r.choices[0].message.content or "").strip()
    tokens = getattr(r.usage, "total_tokens", "?")
    status = "PASS" if "hello" in text.lower() else "WARN"
    return status, f'{r.model} replied "{text[:40]}" in {time.time() - t0:.1f}s ({tokens} tokens)'
check("OpenAI says hello (Virtual Lab path)", _openai_hello, fix=api_hint)

In [ ]:
async def _claude_hello():
    from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock, ResultMessage
    opts = ClaudeAgentOptions(model=CLAUDE_MODEL, max_turns=1,
                              cwd=tempfile.mkdtemp(prefix="preflight-"))
    t0, text, model, cost, error = time.time(), "", None, None, None
    async for m in query(prompt="Reply with exactly two words: hello agent", options=opts):
        if isinstance(m, AssistantMessage):
            model = getattr(m, "model", None) or model
            text += "".join(b.text for b in m.content if isinstance(b, TextBlock))
        elif isinstance(m, ResultMessage):
            cost = getattr(m, "total_cost_usd", None)
            if getattr(m, "is_error", False):
                error = f"{m.subtype}: {str(getattr(m, 'result', ''))[:150]}"
    if error:
        return "FAIL", error
    status = "PASS" if "hello" in text.lower() else "WARN"
    cost_s = f", about US${cost:.4f}" if isinstance(cost, (int, float)) else ""
    return status, f'{model or CLAUDE_MODEL} replied "{text.strip()[:40]}" in {time.time() - t0:.1f}s{cost_s}'

await acheck("Claude says hello (Agent SDK, Virtual Biotech path)", _claude_hello, fix=api_hint, timeout=120)

## 4 · Tools over MCP
Agents call tools through the Model Context Protocol (MCP). The first check starts a tiny MCP server inside this notebook and calls its tool. The second connects to the pre-built Paper2Agent servers you'll use in Module 2.

In [ ]:
async def _mcp_local():
    from fastmcp import FastMCP, Client
    server = FastMCP("preflight")

    @server.tool()
    def echo_gene(symbol: str) -> str:
        """Return the gene symbol: a do-nothing tool for the self-test."""
        return f"{symbol} ok"

    async with Client(server) as c:
        tools = await c.list_tools()
        res = await c.call_tool("echo_gene", {"symbol": TARGET_GENE_SYMBOL})
    out = getattr(res, "data", None)
    if out is None:   # older fastmcp versions return a list of content blocks
        blocks = getattr(res, "content", res)
        out = blocks[0].text
    return "PASS", f"local MCP server listed {len(tools)} tool and returned '{out}'"
await acheck("MCP works on this machine", _mcp_local, fix=HELPER, timeout=60)

async def _mcp_remote(target):
    from fastmcp import Client
    async with Client(str(target)) as c:
        tools = await c.list_tools()
    names = ", ".join(t.name for t in tools[:4]) + ("…" if len(tools) > 4 else "")
    return ("PASS" if tools else "WARN"), f"{len(tools)} tools ({names})"

if not PAPER_AGENT_SERVERS:
    record("Paper2Agent servers (Module 2)", "SKIP", "none listed in Settings")
for name, target in PAPER_AGENT_SERVERS.items():
    await acheck(f"Paper agent: {name}", lambda t=target: _mcp_remote(t), timeout=60,
                 fix="Couldn't connect to this paper agent. It may be restarting; re-run this cell in a minute, then tell a helper.")

## 5 · Public databases
The Virtual Biotech team queries these live in Module 3, and the PubMed check covers the Module 1 stretch task. Each check asks a small question about the workshop target.

In [ ]:
NET_FIX = ("Couldn't reach this database. It may be briefly down or slow. Re-run this cell; "
           "if it keeps failing, tell a helper (cached results are available on the day).")

def _opentargets():
    q = "query($id: String!) { target(ensemblId: $id) { approvedSymbol approvedName } }"
    d = http_json("https://api.platform.opentargets.org/api/v4/graphql",
                  payload={"query": q, "variables": {"id": TARGET_ENSEMBL_ID}})
    t = (d.get("data") or {}).get("target")
    if not t:
        return "FAIL", f"no answer for {TARGET_ENSEMBL_ID}: {str(d)[:120]}"
    return "PASS", f"{t['approvedSymbol']}: {t['approvedName']}"
check("Open Targets", _opentargets, fix=NET_FIX)

def _ctgov():
    d = http_json("https://clinicaltrials.gov/api/v2/studies",
                  params={"query.term": TRIAL_TERM, "query.cond": TRIAL_CONDITION,
                          "countTotal": "true", "pageSize": 1, "fields": "NCTId"})
    return "PASS", f"{d.get('totalCount', '?')} studies match '{TRIAL_TERM}' in {TRIAL_CONDITION}"
check("ClinicalTrials.gov", _ctgov, fix=NET_FIX)

def _pubmed():
    d = http_json("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
                  params={"db": "pubmed", "term": f"{TARGET_GENE_SYMBOL} AND {TRIAL_CONDITION}",
                          "retmode": "json", "retmax": 0, "tool": "workshop_preflight"})
    return "PASS", f"{d['esearchresult']['count']} PubMed records for {TARGET_GENE_SYMBOL} AND {TRIAL_CONDITION}"
check("PubMed (E-utilities)", _pubmed, fix=NET_FIX, on_error="WARN")

def _chembl():
    d = http_json("https://www.ebi.ac.uk/chembl/api/data/target/search.json",
                  params={"q": TARGET_GENE_SYMBOL, "limit": 1})
    return "PASS", f"{d.get('page_meta', {}).get('total_count', '?')} ChEMBL targets match {TARGET_GENE_SYMBOL}"
check("ChEMBL", _chembl, fix=NET_FIX, on_error="WARN")

def _census():
    import cellxgene_census
    desc = cellxgene_census.get_census_version_description("stable")
    with cellxgene_census.open_soma(census_version="stable") as census:
        s = census["census_info"]["summary"].read().concat().to_pandas()
    total = s.set_index("label").loc["total_cell_count", "value"]
    return "PASS", f"Census {desc.get('release_build', 'stable')} opened ({int(total):,} cells)"
check("CELLxGENE Census", _census, fix=NET_FIX)

## 6 · Workshop data

In [ ]:
def _dataset():
    import anndata as ad
    a = ad.read_h5ad(DATASET_PATH, backed="r")   # backed mode: reads metadata only, light on memory
    try:
        names = set(map(str, a.var_names))
        for col in ("feature_name", "gene_symbols", "gene_symbol", "symbol"):
            if col in a.var.columns:
                names |= set(map(str, a.var[col]))
        found = TARGET_GENE_SYMBOL in names or any(n.split(".")[0] == TARGET_ENSEMBL_ID for n in names)
        detail = (f"{a.n_obs:,} cells × {a.n_vars:,} genes; "
                  f"{TARGET_GENE_SYMBOL} {'is present' if found else 'NOT found in the gene list'}")
    finally:
        a.file.close()
    return ("PASS" if found else "WARN"), detail
check("Workshop dataset loads", _dataset, fix=HELPER)

## Result

In [ ]:
counts = {s: sum(r["status"] == s for r in RESULTS) for s in ICONS}
ready = counts["FAIL"] == 0
host = socket.gethostname()
line = (f"PREFLIGHT {'READY' if ready else 'NOT READY'} | {counts['PASS']} pass · {counts['WARN']} warn · "
        f"{counts['FAIL']} fail · {counts['SKIP']} skipped | {host} | {time.strftime('%Y-%m-%d %H:%M')}")

attention = [r for r in RESULTS if r["status"] in ("FAIL", "WARN")]
if attention:
    print("Needs a look:\n")
    for r in sorted(attention, key=lambda r: r["status"] != "FAIL"):
        print(f"  {ICONS[r['status']]} {r['check']}: {r['detail']}")
        if r["fix"]:
            print(f"      ↳ {r['fix']}")
    print()

try:
    out = Path.home() / "preflight_result.json"
    out.write_text(json.dumps({"summary": line, "host": host, "results": RESULTS}, indent=2))
except OSError:
    pass

bar = "=" * len(line)
print(bar, line, bar, sep="\n")
print("\nYou're all set. See you at the workshop!" if ready else
      "\nPlease reply to the setup email with the PREFLIGHT line above (or show it to a helper on the day).")

---
**For organisers.** To check every instance before the day without opening VS Code, run this notebook headless on each one:

```
jupyter nbconvert --to notebook --execute --inplace 00_preflight.ipynb
```

Each run writes `~/preflight_result.json`, so you can collect the `summary` lines from all instances in one sweep.